# Davinci — Pretraining Data Loader & Training Engine

This notebook connects the prepared corpus, SentencePiece tokenizer, and PyTorch model into a resumable causal-language-model pretraining loop.

**Run in stages.** First configure paths and run the data/tokenizer smoke tests. Then run a short pilot (for example 20–100 optimizer steps). Do not launch a long run until token counts, validation loss, GPU memory, checkpoint recovery, and throughput have been checked.

This notebook expects the model implementation from Notebook 5 to be available as `davinci_model.py` or pasted/imported into the runtime. It deliberately does not silently recreate a potentially divergent architecture.

## 1. Runtime and dependencies

Colab usually includes PyTorch. Install only the lightweight data/tokenizer dependencies; avoid replacing the Colab CUDA-enabled PyTorch build.

In [ ]:
!pip -q install sentencepiece tqdm

In [ ]:
import os, json, math, time, random, hashlib, shutil
from pathlib import Path
from collections import deque
import numpy as np
import sentencepiece as spm
from tqdm.auto import tqdm
import torch
from torch.utils.data import IterableDataset, DataLoader
import torch.nn.functional as F

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch",torch.__version__,"device",DEVICE)
if DEVICE=="cuda":
    print(torch.cuda.get_device_name(0), "VRAM GiB",round(torch.cuda.get_device_properties(0).total_memory/2**30,2))

## 2. Configure inputs and output directory

Mount Drive if you want data and checkpoints to survive a Colab runtime reset. Paths are intentionally explicit: files from earlier notebooks do not automatically exist in a fresh runtime.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# EDIT THESE PATHS
CORPUS_JSONL = "/content/drive/MyDrive/Davinci/data/pretraining.jsonl"
TOKENIZER_MODEL = "/content/drive/MyDrive/Davinci/tokenizer/davinci_unigram_32000.model"
OUTPUT_DIR = Path("/content/drive/MyDrive/Davinci/pretraining_runs/run_001")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# Input format: "text" for one text field, "messages" for chat-style messages.
INPUT_MODE = "text"
TEXT_FIELD = "text"
MESSAGES_FIELD = "messages"
MIN_TOKENS_PER_DOCUMENT = 8

# Deterministic document-level split; records are assigned by stable hash.
VALIDATION_FRACTION = 0.01
SEQ_LEN = 1024
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_WORKERS = 0
assert INPUT_MODE in {"text","messages"}
assert 0 < VALIDATION_FRACTION < 0.5
print("Output:",OUTPUT_DIR)

## 3. Load and validate the tokenizer

In [ ]:
assert Path(TOKENIZER_MODEL).is_file(), f"Tokenizer not found: {TOKENIZER_MODEL}"
sp=spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL)
VOCAB_SIZE=sp.get_piece_size()
print("Tokenizer vocab:",VOCAB_SIZE,"EOS:",sp.eos_id(),"UNK:",sp.unk_id())
assert VOCAB_SIZE>0 and sp.eos_id()>=0
sample="این یک آزمون برای مدل داوینچی است. This is a bilingual test."
ids=sp.encode(sample,out_type=int)
print("sample tokens:",len(ids),"roundtrip:",sp.decode(ids))
assert len(ids)>0

## 4. Inspect corpus records before training

This is a bounded inspection pass. It does not load the entire corpus into RAM. The selected text/message field must match the actual JSONL schema.

In [ ]:
CORPUS=Path(CORPUS_JSONL)
assert CORPUS.is_file(), f"Corpus file not found: {CORPUS}"
def extract_document(obj):
    if INPUT_MODE=="text":
        value=obj.get(TEXT_FIELD)
        if isinstance(value,str): return value.strip()
        return ""
    msgs=obj.get(MESSAGES_FIELD)
    if not isinstance(msgs,list): return ""
    parts=[]
    for m in msgs:
        if isinstance(m,dict):
            role=str(m.get("role","user"))
            content=m.get("content","")
            if isinstance(content,str) and content.strip():
                parts.append(f"{role}: {content.strip()}")
    return "\n".join(parts)

valid=0; invalid=0; examples=[]
with CORPUS.open("r",encoding="utf-8") as f:
    for line_no,line in enumerate(f):
        if not line.strip(): continue
        try:
            obj=json.loads(line); text=extract_document(obj)
            if text: valid+=1
            else: invalid+=1
            if text and len(examples)<3: examples.append(text[:300])
        except Exception: invalid+=1
        if line_no>=9999: break
print({"scanned_max":10000,"valid":valid,"invalid_or_empty":invalid})
for e in examples: print("\n---\n",e)

## 5. Streaming token-block dataset

The iterable dataset reads JSONL incrementally, assigns each document deterministically to train or validation, tokenizes on demand, appends EOS, and yields fixed-length blocks. A small bounded shuffle buffer randomizes document order without loading the corpus into RAM.

For reproducibility, keep `NUM_WORKERS=0` for the first pilot. Multi-worker sharding is implemented, but verify that every record is visited exactly once before increasing workers.

In [ ]:
class PackedJSONLDataset(IterableDataset):
    def __init__(self,path,sp,split,seq_len,seed=42,val_fraction=0.01,shuffle_buffer=256):
        super().__init__()
        self.path=str(path); self.sp=sp; self.split=split
        self.seq_len=seq_len; self.seed=seed
        self.val_fraction=val_fraction; self.shuffle_buffer=shuffle_buffer
    def _belongs(self,line):
        h=int(hashlib.sha256(line.encode("utf-8")).hexdigest()[:16],16)/float(16**16)
        return (h<self.val_fraction) if self.split=="validation" else (h>=self.val_fraction)
    def _documents(self):
        with open(self.path,"r",encoding="utf-8") as f:
            for line in f:
                if not line.strip() or not self._belongs(line): continue
                try:
                    obj=json.loads(line); text=extract_document(obj)
                    if text: yield text
                except Exception:
                    continue
    def __iter__(self):
        worker=torch.utils.data.get_worker_info()
        worker_id=worker.id if worker else 0
        workers=worker.num_workers if worker else 1
        rng=random.Random(self.seed+worker_id+(0 if self.split=="train" else 100000))
        # Deterministic worker sharding by document ordinal.
        buffer=[]
        token_buffer=[]
        for ordinal,doc in enumerate(self._documents()):
            if ordinal%workers!=worker_id: continue
            if self.split=="train" and self.shuffle_buffer>1:
                if len(buffer)<self.shuffle_buffer: buffer.append(doc); continue
                j=rng.randrange(len(buffer)); doc,buffer[j]=buffer[j],doc
            ids=self.sp.encode(doc,out_type=int)
            if not ids: continue
            token_buffer.extend(ids)
            eos=self.sp.eos_id()
            if eos>=0: token_buffer.append(eos)
            while len(token_buffer)>=self.seq_len+1:
                block=token_buffer[:self.seq_len+1]
                del token_buffer[:self.seq_len+1]
                yield torch.tensor(block,dtype=torch.long)
        for doc in buffer:
            ids=self.sp.encode(doc,out_type=int)
            token_buffer.extend(ids)
            if self.sp.eos_id()>=0: token_buffer.append(self.sp.eos_id())
            while len(token_buffer)>=self.seq_len+1:
                block=token_buffer[:self.seq_len+1]; del token_buffer[:self.seq_len+1]
                yield torch.tensor(block,dtype=torch.long)

def collate_blocks(items):
    return torch.stack(items,dim=0)

train_ds=PackedJSONLDataset(CORPUS,sp,"train",SEQ_LEN,SEED,VALIDATION_FRACTION)
val_ds=PackedJSONLDataset(CORPUS,sp,"validation",SEQ_LEN,SEED,VALIDATION_FRACTION,shuffle_buffer=1)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,collate_fn=collate_blocks)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,collate_fn=collate_blocks)
first=next(iter(train_loader))
print("batch shape",tuple(first.shape),"tokens",first.numel())

## 6. Import the model implementation

If you downloaded Notebook 5's implementation as a Python module, set `MODEL_MODULE_PATH` and import it. The notebook also supports a local file written from the implementation notebook. Do not proceed with a different architecture/config by accident.

In [ ]:
# Import the canonical implementation from this repository.
import os, sys, subprocess, base64
REPO_DIR=Path('/content/Davinci')
if not (REPO_DIR/'.git').exists():
    from google.colab import userdata
    token=userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Add a GitHub fine-grained token with Contents: read permission as Colab secret GITHUB_TOKEN, then rerun this cell.')
    extra=base64.b64encode(('x-access-token:'+token).encode()).decode()
    subprocess.run(['git','-c',f'http.extraheader=AUTHORIZATION: basic {extra}','clone','--depth','1','https://github.com/keyhmoham/Davinci.git',str(REPO_DIR)],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci import DavinciConfig, DavinciForCausalLM
from src.davinci.train import train_microbatch, evaluate_loss
print('Imported canonical model from',REPO_DIR)


## 7. Configure model and verify tokenizer vocabulary

Use the exact frozen tokenizer vocabulary size. This config must otherwise match the architecture decision in Notebook 4 and the model class implementation.

In [ ]:
MODEL_CONFIG={
 "vocab_size":VOCAB_SIZE,"hidden_size":1280,"num_hidden_layers":28,
 "num_attention_heads":20,"num_key_value_heads":4,"head_dim":64,
 "intermediate_size":3328,"norm_eps":1e-5,"rope_theta":10000.0,
"local_window":4096,
 "global_every_n_layers":4,"tie_word_embeddings":True,"dropout":0.0,
 "gradient_checkpointing":True
}
assert MODEL_CONFIG["vocab_size"]==sp.get_piece_size()
assert MODEL_CONFIG["hidden_size"]==MODEL_CONFIG["num_attention_heads"]*MODEL_CONFIG["head_dim"]
print("Analytical parameter count is computed before allocating the full model.")
def count_parameters_formula(c):
    d=c["hidden_size"]; kv=c["num_key_value_heads"]; hd=c["head_dim"]
    return c["num_hidden_layers"]*(2*d*d+2*kv*hd*d+3*d*c["intermediate_size"]+2*d)+c["vocab_size"]*d+d
print(f"{count_parameters_formula(MODEL_CONFIG):,} parameters")


## 8. Training configuration — begin with a pilot

A 500M model is not expected to train comfortably on a typical free Colab GPU. Start with a small model config or a short low-memory pilot. Do not interpret a successful smoke run as evidence that full-scale pretraining fits.

In [ ]:
MAX_STEPS=50                 # pilot only; raise after validating everything
WARMUP_STEPS=5
LEARNING_RATE=3e-4
MIN_LR=3e-5
WEIGHT_DECAY=0.1
MAX_GRAD_NORM=1.0
EVAL_EVERY=25
SAVE_EVERY=25
LOG_EVERY=5
VAL_BATCHES=10
USE_BF16=(DEVICE=="cuda" and torch.cuda.is_bf16_supported())
AMP_DTYPE=torch.bfloat16 if USE_BF16 else torch.float16
print({"bf16":USE_BF16,"amp_dtype":str(AMP_DTYPE),"effective_batch_size":BATCH_SIZE*GRAD_ACCUM_STEPS})

## 9. Build model, optimizer, and scheduler

For the first run, consider changing `MODEL_CONFIG` to a smaller configuration compatible with the imported model class. AdamW optimizer states can require substantially more memory than model weights alone.

In [ ]:
model=DavinciForCausalLM(MODEL_CONFIG).to(DEVICE)
n_params=sum(p.numel() for p in model.parameters())
print("Parameters:",f"{n_params:,}")
assert n_params==count_parameters_formula(MODEL_CONFIG), "Model/config parameter count mismatch"

try:
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY,fused=(DEVICE=="cuda"))
except (TypeError,RuntimeError):
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)

def lr_at(step):
    if step<WARMUP_STEPS: return LEARNING_RATE*(step+1)/max(1,WARMUP_STEPS)
    progress=min(1.0,(step-WARMUP_STEPS)/max(1,MAX_STEPS-WARMUP_STEPS))
    cosine=0.5*(1+math.cos(math.pi*progress))
    return MIN_LR+(LEARNING_RATE-MIN_LR)*cosine
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s:lr_at(s)/LEARNING_RATE)
scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE=="cuda" and AMP_DTYPE==torch.float16))
print("Gradient checkpointing:",model.config.gradient_checkpointing,"GradScaler:",scaler.is_enabled())

## 10. Evaluation and checkpoint helpers

In [ ]:
@torch.no_grad()
def evaluate(model,loader,max_batches=10):
    was_training=model.training
    model.eval(); losses=[]
    it=iter(loader)
    try:
        for _ in range(max_batches):
            try: tokens=next(it)
            except StopIteration: break
            tokens=tokens.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type="cuda",dtype=AMP_DTYPE,enabled=(DEVICE=="cuda")):
                out=model(tokens,labels=tokens)
            if out.get("loss") is not None: losses.append(float(out["loss"].float().cpu()))
    finally:
        model.train(was_training)
    return float(np.mean(losses)) if losses else None

def tokenizer_sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def save_checkpoint(step,model,optimizer,scheduler,scaler,loss,micro_step):
    path=OUTPUT_DIR/f"checkpoint_step_{step:08d}.pt"
    state={
        "step":step,"micro_step":micro_step,"model":model.state_dict(),"optimizer":optimizer.state_dict(),
        "scheduler":scheduler.state_dict(),"scaler":scaler.state_dict(),"loss":loss,
        "model_config":MODEL_CONFIG,"tokenizer_model":TOKENIZER_MODEL,
        "tokenizer_sha256":tokenizer_sha256(TOKENIZER_MODEL),"seq_len":SEQ_LEN,"seed":SEED,
        "python_rng":random.getstate(),"numpy_rng":np.random.get_state(),
        "torch_rng":torch.get_rng_state(),
        "cuda_rng":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    torch.save(state,path)
    latest=OUTPUT_DIR/"latest_checkpoint.pt"
    shutil.copy2(path,latest)
    return path

def load_checkpoint(path,model,optimizer,scheduler,scaler):
    # Only load checkpoints created by this trusted training run: this uses pickle-backed state.
    state=torch.load(path,map_location=DEVICE,weights_only=False)
    if state["model_config"]!=MODEL_CONFIG:
        raise ValueError("Checkpoint model config differs from current config")
    if state.get("tokenizer_sha256")!=tokenizer_sha256(TOKENIZER_MODEL):
        raise ValueError("Checkpoint tokenizer fingerprint differs from current tokenizer file")
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    if "scaler" in state: scaler.load_state_dict(state["scaler"])
    random.setstate(state["python_rng"])
    np.random.set_state(state["numpy_rng"])
    torch.set_rng_state(state["torch_rng"].cpu())
    if torch.cuda.is_available() and state.get("cuda_rng") is not None:
        torch.cuda.set_rng_state_all(state["cuda_rng"])
    return int(state["step"]),state.get("loss"),int(state.get("micro_step",0))

## 11. Training loop

The loop uses gradient accumulation, autocast, gradient clipping, a learning-rate schedule, periodic validation, and resumable checkpoints. It stops cleanly if the iterable corpus is exhausted.

In [ ]:
RESUME_FROM = None  # e.g. str(OUTPUT_DIR/"latest_checkpoint.pt")
global_step=0; micro_step=0
if RESUME_FROM:
    global_step,last_loss,micro_step=load_checkpoint(RESUME_FROM,model,optimizer,scheduler,scaler)
    print("Resumed at optimizer step",global_step)

model.train()
optimizer.zero_grad(set_to_none=True)
running=[]; start=time.time(); seen_tokens=0
train_iter=iter(train_loader)
# Recreate the deterministic iterable position after a checkpoint (keep NUM_WORKERS=0 for exact replay).
for _ in range(micro_step):
    try: next(train_iter)
    except StopIteration: raise RuntimeError("Checkpoint data cursor exceeds the current training split.")
progress=tqdm(total=MAX_STEPS,initial=global_step)
while global_step<MAX_STEPS:
    try:
        tokens=next(train_iter)
    except StopIteration:
        print("Training iterator exhausted; increase corpus or inspect split/sequence length.")
        break
    result=train_microbatch(
        model,tokens,optimizer,scaler=scaler,amp_dtype=AMP_DTYPE,
        max_grad_norm=MAX_GRAD_NORM,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        microbatch_index=micro_step,
    )
    running.append(float(result["loss"].cpu()))
    seen_tokens+=int(tokens.numel()); micro_step+=1
    if result["optimizer_step"]:
        scheduler.step()
        global_step+=1; progress.update(1)
        if global_step%LOG_EVERY==0:
            mean_loss=float(np.mean(running[-LOG_EVERY:]))
            elapsed=max(time.time()-start,1e-6)
            progress.set_postfix(loss=f"{mean_loss:.4f}",lr=f"{scheduler.get_last_lr()[0]:.2e}",tok_s=f"{seen_tokens/elapsed:.0f}")
        if EVAL_EVERY and global_step%EVAL_EVERY==0:
            val_loss=evaluate_loss(model,val_loader,max_batches=VAL_BATCHES,amp_dtype=AMP_DTYPE)
            print(f"\nstep={global_step} validation_loss={val_loss}")
        if SAVE_EVERY and global_step%SAVE_EVERY==0:
            path=save_checkpoint(global_step,model,optimizer,scheduler,scaler,float(np.mean(running[-LOG_EVERY:])),micro_step)
            print("\nSaved",path)
progress.close()
print("Finished at optimizer step",global_step)
if micro_step%GRAD_ACCUM_STEPS:
    print("WARNING: final incomplete accumulation window was not stepped; discard it before resuming.")

## 12. Final save and run manifest

In [ ]:
final_loss=float(np.mean(running[-min(len(running),LOG_EVERY):])) if running else None
final_path=save_checkpoint(global_step,model,optimizer,scheduler,scaler,final_loss,micro_step)
manifest={
 "global_step":global_step,"final_recent_loss":final_loss,
 "parameter_count":n_params,"model_config":MODEL_CONFIG,
 "tokenizer_model":TOKENIZER_MODEL,
 "tokenizer_sha256":tokenizer_sha256(TOKENIZER_MODEL),"corpus_jsonl":CORPUS_JSONL,
 "input_mode":INPUT_MODE,"sequence_length":SEQ_LEN,
 "batch_size":BATCH_SIZE,"gradient_accumulation_steps":GRAD_ACCUM_STEPS,
 "effective_batch_size":BATCH_SIZE*GRAD_ACCUM_STEPS,
 "device":DEVICE,"amp_dtype":str(AMP_DTYPE),"seed":SEED,
 "note":"Pilot configuration. Record dataset provenance and exact code commit separately."
}
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Checkpoint:",final_path)
print(json.dumps(manifest,indent=2))

## 13. Operational checks before scaling

- Verify the train and validation split are non-empty and sufficiently large.
- Check that the source JSONL is the intended pretraining corpus, not an unreviewed mixture of instruction/agent data.
- Confirm tokenizer/model vocabulary sizes match exactly.
- Run a tiny overfit test on a handful of batches; loss should fall.
- Kill and resume a pilot to verify optimizer/scheduler recovery.
- Monitor peak VRAM and host RAM, tokens/sec, data-loader stalls, and validation loss.
- Save the exact dataset manifest, tokenizer model, model config, notebook commit, and training logs with every run.
- Do not increase context length to 500K in this notebook. Long-context curriculum and efficient attention are separate engineering stages.

**Known limitations:** no packed document-boundary loss masking, no weighted source mixing, no deduplication at token-block level, gradient checkpointing is now available but increases compute time, no distributed training, and no automatic data-quality scoring. Add and test these before serious pretraining.